# Identification best performing random forest model parameters

In [ ]:
# Libraries
import os, shutil
import numpy as np
import pandas as pd
import dask.dataframe as dd
from dask.distributed import wait
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from dask_ml.model_selection import GridSearchCV

In [ ]:
# Directories
dir02 = '../paper_ecosystem_carbon/output/02_database_prep/'
dir03 = '../paper_ecosystem_carbon/output/03_rf_predict/'

---

In [ ]:
# Initialize dask
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(queue='compute',
                       cores=48,
                       memory='256 GB',
                       account='bm0891', 
                       interface="ib0",
                       walltime='04:00:00',
                       local_directory='../dask/',
                       job_extra_directives=[
                           '-o ../dask/LOG_worker_%j.o',
                           '-e ../dask/LOG_worker_%j.o'])

# Scale dask clusters
cluster.scale(jobs=15) #15

# Configurate dashboard url
import dask
dask.config.config.get('distributed').get('dashboard') \
    .update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})

# Create client
from dask.distributed import Client
client = Client(cluster)
client

In [ ]:
# Get database
dd_base = dd.read_parquet(dir02 + 'dd_base_r001.parquet')

In [ ]:
# List with ecosystem types
list_etype = ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
              'temp_dnf', 'glob_shrub', 'glob_grass', 'glob_tundr', 
              'glob_mount', 'glob_crop', 'glob_past']

# List with covariates
vars_soilgrids2017 = ['soilgrids2017_bdricm', 'soilgrids2017_bdrlog',
                      'soilgrids2017_bdticm']

vars_soilgrids2020 = ['soilgrids2020_cec', 'soilgrids2020_cfvo', 
                      'soilgrids2020_clay', 'soilgrids2020_phh2o', 
                      'soilgrids2020_sand', 'soilgrids2020_silt']

vars_worldclim = ['worldclim_bio1', 'worldclim_bio3', 'worldclim_bio4', 
                  'worldclim_bio5', 'worldclim_bio6', 'worldclim_bio12', 
                  'worldclim_bio13', 'worldclim_bio14', 'worldclim_bio15', 
                  'worldclim_elev']

vars_geom90m = ['geom90m_convergence', 'geom90m_cti', 'geom90m_eastness',
               'geom90m_northness', 'geom90m_slope', 'geom90m_spi']

list_var_exp = [*vars_soilgrids2017, *vars_soilgrids2020, *vars_worldclim, 
                *vars_geom90m]

In [ ]:
# Create directory to store parameter files
# Directory path and string
dir_params_r001 = f'{dir03}files_params_r001'

# Delete directory if exists
if os.path.isdir(dir_params_r001):
    shutil.rmtree(dir_params_r001)

# Create directory
os.makedirs(dir_params_r001)

In [ ]:
def random_forest_params_rank(ctype, etype):
   
    """Find 10 best parameters for random forest regression in 
       pre-defined parameter space
    """

    # Read training data
    # Repartition and shuffle data
    df_train = dd_base[dd_base[f'train_{ctype}_{etype}'] == True] \
        .repartition(partition_size='1000 MiB') \
        .sample(frac=1, random_state=42) \
        .persist()

    wait(df_train)
    
    # Define x_train
    X_train = df_train[list_var_exp].to_dask_array(lengths=True)
    
    # Define y_train (ensure consistency with 02_05_prep_database_r001)
    if ctype == 'cveg':
        if etype in ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
                     'temp_dnf', 'glob_shrub']:
            cbase = 'cveg_biom'    
        if etype in ['glob_tundr', 'glob_mount']:
            cbase = 'cveg_tund'
        if etype in ['glob_grass', 'glob_past']:
            cbase = 'cveg_gras'
        if etype == 'glob_crop':
            cbase = 'cveg_crop'
    if ctype == 'csoil':
        cbase = 'csoil_000_100'
    
    y_train = df_train[cbase].to_dask_array(lengths=True)
    
    # Create pipline for randomforest regressor, needed for gridserachcv
    pipe = make_pipeline(
        RandomForestRegressor(random_state=42),
        )
            
    # Select parameters of grid
    param_grid = dict(
        #randomforestregressor__max_depth = [50],
        randomforestregressor__n_estimators = [100, 200, 300], #[50, 100, 150],
        randomforestregressor__min_samples_leaf = [2, 3, 4, 5],
        randomforestregressor__max_features = [2, 3, 4, 5, 6, 7, 8, 9, 10],
        randomforestregressor__random_state = [42]
        )
        
    # Grid search with cross validation
    grid_results = GridSearchCV(pipe, param_grid, 
        cv=5, # five-fold cross validation
        scoring='r2', # r2 as evaluation criteria # sklearn: R2 gives the same ranking as squared error.
        refit=False,
        scheduler=client,
        cache_cv=False,
        error_score='raise')
            
    # Fit models
    grid_results_fit = grid_results.fit(X_train, y_train)
    
    # Extract rank_test_score and parameters
    df_params_rank = pd.DataFrame(grid_results_fit.cv_results_) \
        [
        'rank_test_score',
        'param_randomforestregressor__min_samples_leaf',
        'param_randomforestregressor__max_features',
        'param_randomforestregressor__n_estimators',
        'param_randomforestregressor__random_state',
        ] \
        .sort_values('rank_test_score') \
        .reset_index(drop=True)
            
    # Rename columns
    df_params_rank.columns = [
        i[29:] if i.startswith('param_randomforestregressor__') 
        else i for i in df_params_rank.columns
        ]
        
    # Export
    df_params_rank.to_csv(
        f'{dir03}files_params_r001/df_params_rank_{ctype}_{etype}.csv', 
        index=False)

In [ ]:
# Get parameters for cveg
for etype in list_etype:
    ctype = 'cveg'
    print(f'{ctype}_{etype}')
    %time random_forest_params_rank(ctype, etype)

In [ ]:
## Get parameters for csoil
for etype in list_etype:
    ctype = 'csoil'
    print(f'{ctype}_{etype}')
    %time random_forest_params_rank(ctype, etype)

In [ ]:
cluster.close()